In [ ]:
import sys
from pathlib import Path
carpeta_pipeline = next(ruta for ruta in [Path.cwd(), Path.cwd() / 'pipeline_ingesta', Path.cwd() / 'notebook/unidad_IV/pipeline_ingesta'] if (ruta / 'auditoria_pipeline.py').is_file())
sys.path.insert(0, str(carpeta_pipeline.resolve()))
from auditoria_pipeline import iniciar_etapa

auditoria_etapa = iniciar_etapa('02_profiling', 'invoices')

# 02 - Profiling y Control de Calidad de Invoices

Valida campos obligatorios, identificadores enteros positivos, totales no negativos, fechas y `IsCreditNote` de las cabeceras de factura. Los campos opcionales, como `OrderID`, pueden ser nulos.

**Entrada:** `raw.invoices`, cargada por `01_extraccion_raw_Invoices.ipynb`.

**Salida:** Reportes `reporte_calidad_invoices_*.csv` y `.json` en `datos/bronze/` y errores en `raw.validacion_calidad_invoices`.

Ejecutar despues de la extraccion, siguiendo los pasos en orden.

In [1]:
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, BD_DESTINO
from datetime import datetime
import os
import pandas as pd
import json

engine_origen, engine_destino = obtener_engines()

# Rutas
DATOS_DIR = './datos'
BRONZE_DIR = os.path.join(DATOS_DIR, 'bronze')
os.makedirs(BRONZE_DIR, exist_ok=True)

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Timestamp: {TIMESTAMP}')
print(f'Destino: {BD_DESTINO}')

Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Timestamp: 20261103_124046
Destino: WideWorldImportersDW2


## Paso 1: Crear tabla de validación

In [ ]:
from sqlalchemy import text

sql_crear_validacion = """
IF OBJECT_ID('raw.validacion_calidad_invoices', 'U') IS NULL
BEGIN
    CREATE TABLE raw.validacion_calidad_invoices (
        id_validacion       INT PRIMARY KEY IDENTITY(1,1),
        id_raw              INT NOT NULL,
        lote_id             NVARCHAR(50) NOT NULL,
        tipo_error          NVARCHAR(100) NOT NULL,
        descripcion         NVARCHAR(500),
        valor_campo         NVARCHAR(MAX),
        campo_afectado      NVARCHAR(100),
        timestamp_validacion DATETIME DEFAULT GETDATE(),
        es_critico          BIT DEFAULT 0,
        INDEX IX_raw (id_raw),
        INDEX IX_lote (lote_id)
    );
END;
"""

try:
    with engine_destino.begin() as conn:
        conn.execute(text(sql_crear_validacion))
    print('[OK] Tabla raw.validacion_calidad_invoices creada')
except Exception as error:
    print(f'[ERROR] {error}')

[OK] Tabla raw.validacion_calidad creada


## Paso 2: Leer datos RAW

In [ ]:
try:
    df_raw = pd.read_sql_query(
        """
        SELECT
            id_raw, lote_id, numero_fila, payload, hash_sha256
        FROM raw.invoices
        ORDER BY id_raw DESC
        """,
        engine_destino
    )
    print(f'[OK] Se leyeron {len(df_raw)} registros RAW de Invoices')
    print(f'Lotes unicos: {df_raw["lote_id"].nunique()}')
    print('\nUltimos lotes:')
    print(df_raw[['lote_id', 'numero_fila']].drop_duplicates('lote_id'))
except Exception as error:
    print(f'[ERROR] {error}')
    df_raw = None

[OK] Se leyeron 100 registros RAW
Lotes únicos: 1

Últimos lotes:
                lote_id  numero_fila
0  LOTE_20261103_112725          100


## Paso 3: Parsear y validar payloads

In [ ]:
errores_validacion = []
registros_validos = []

campos_id = [
    'InvoiceID', 'CustomerID', 'BillToCustomerID', 'DeliveryMethodID',
    'ContactPersonID', 'AccountsPersonID', 'SalespersonPersonID',
    'PackedByPersonID', 'LastEditedBy'
]
campos_totales = ['TotalDryItems', 'TotalChillerItems']
campos_fecha = ['InvoiceDate', 'LastEditedWhen']
campos_requeridos = campos_id + campos_totales + campos_fecha + ['IsCreditNote']

if df_raw is not None and len(df_raw) > 0:
    for idx, row in df_raw.iterrows():
        id_raw = int(row['id_raw'])
        lote_id = row['lote_id']
        errores_fila = []

        try:
            payload = json.loads(row['payload'])
            if not isinstance(payload, dict):
                raise ValueError('El payload debe ser un objeto JSON')
        except (ValueError, TypeError) as error:
            errores_validacion.append({
                'id_raw': id_raw,
                'lote_id': lote_id,
                'tipo_error': 'JSON_INVALIDO',
                'descripcion': f'Payload JSON no valido: {error}',
                'campo_afectado': 'payload',
                'valor_campo': str(row['payload'])[:100],
                'es_critico': 1
            })
            continue

        for campo in campos_requeridos:
            valor = payload.get(campo)
            if valor is None or str(valor).strip() == '':
                errores_fila.append((campo, 'NULO_EN_CAMPO_REQUERIDO',
                                     f'Campo requerido {campo} es nulo o vacio'))
                continue

            if campo in campos_id or campo in campos_totales:
                minimo = 1 if campo in campos_id else 0
                try:
                    numero = float(valor)
                    valido = not isinstance(valor, bool) and numero.is_integer() and numero >= minimo
                except (TypeError, ValueError, OverflowError):
                    valido = False
                if not valido:
                    errores_fila.append((campo, 'ENTERO_INVALIDO',
                                         f'{campo} debe ser entero mayor o igual a {minimo}'))
            elif campo in campos_fecha:
                if not isinstance(valor, str) or pd.isna(pd.to_datetime(valor, errors='coerce')):
                    errores_fila.append((campo, 'FECHA_INVALIDA',
                                         f'{campo} no contiene una fecha valida'))
            elif campo == 'IsCreditNote':
                if str(valor).lower() not in ('true', 'false', '0', '1'):
                    errores_fila.append((campo, 'BOOLEANO_INVALIDO',
                                         'IsCreditNote debe ser True, False, 0 o 1'))

        for campo, tipo_error, descripcion in errores_fila:
            errores_validacion.append({
                'id_raw': id_raw,
                'lote_id': lote_id,
                'tipo_error': tipo_error,
                'descripcion': descripcion,
                'campo_afectado': campo,
                'valor_campo': str(payload.get(campo)),
                'es_critico': 1
            })

        if not errores_fila:
            registros_validos.append(payload)

    print('[OK] Validacion completada')
    print(f'Registros validos: {len(registros_validos)}')
    print(f'Errores encontrados: {len(errores_validacion)}')
    if errores_validacion:
        df_errores = pd.DataFrame(errores_validacion)
        print(df_errores[['tipo_error', 'es_critico']].value_counts())
else:
    print('[AVISO] No hay registros RAW para validar')

[OK] Validación completada
Registros válidos: 100
Errores encontrados: 1

Detalle de errores:
tipo_error       es_critico
PRECIO_NEGATIVO  1             1
Name: count, dtype: int64


## Paso 4: Guardar reporte en archivo

In [ ]:
if errores_validacion:
    df_errores = pd.DataFrame(errores_validacion)

    archivo_errores = os.path.join(BRONZE_DIR, f'reporte_calidad_invoices_{TIMESTAMP}.csv')
    df_errores.to_csv(archivo_errores, index=False, encoding='utf-8')
    print(f'[OK] Reporte guardado: {archivo_errores}')

    archivo_errores_json = os.path.join(BRONZE_DIR, f'reporte_calidad_invoices_{TIMESTAMP}.json')
    with open(archivo_errores_json, 'w', encoding='utf-8') as archivo:
        json.dump(errores_validacion, archivo, indent=2, ensure_ascii=False)
    print(f'[OK] Reporte JSON guardado: {archivo_errores_json}')
else:
    print('[OK] Sin errores de validacion')

[OK] Reporte guardado: ./datos\bronze\reporte_calidad_20261103_124046.csv
[OK] Reporte JSON guardado: ./datos\bronze\reporte_calidad_20261103_124046.json


## Paso 5: Insertar errores en BD

In [ ]:
if errores_validacion:
    df_errores = pd.DataFrame(errores_validacion)

    try:
        df_errores.to_sql(
            'validacion_calidad_invoices',
            engine_destino,
            schema='raw',
            if_exists='append',
            index=False,
            chunksize=100
        )
        print(f'[OK] {len(df_errores)} errores registrados en raw.validacion_calidad_invoices')
    except Exception as error:
        print(f'[ERROR] {error}')
else:
    print('[OK] No hay errores para registrar')

[OK] 1 errores registrados en raw.validacion_calidad


## Paso 6: Estadísticas y Resumen

In [ ]:
try:
    sql_resumen = """
    SELECT
        COUNT(*) as total_errores,
        SUM(CASE WHEN es_critico = 1 THEN 1 ELSE 0 END) as errores_criticos,
        COUNT(DISTINCT lote_id) as lotes_afectados,
        COUNT(DISTINCT tipo_error) as tipos_error
    FROM raw.validacion_calidad_invoices
    """

    with engine_destino.connect() as conn:
        result = conn.execute(text(sql_resumen)).mappings().first()

    print('[OK] Resumen de validacion de Invoices en BD:')
    if result:
        for key, val in result.items():
            print(f'  {key}: {val}')
except Exception as error:
    print(f'[ERROR] {error}')

[OK] Resumen de validación en BD:
  total_errores: 1
  errores_criticos: 1
  lotes_afectados: 1
  tipos_error: 1


In [ ]:
auditoria_etapa.finalizar(globals())